# Data Matters Product DS Analysis

This notebook reads a local Supabase CSV export, performs data-quality checks, builds a session funnel, and creates aggregate Product DS outputs.

**No production numbers are embedded in this notebook.**

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = Path('../data/analytics_events.csv')
OUTPUT_DIR = Path('../outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Missing {DATA_PATH}. Run sql/99_export_for_notebook.sql in Supabase and export the result.'
    )

In [ ]:
events = pd.read_csv(DATA_PATH)
required = {
    'id', 'occurred_at', 'session_id', 'event_name', 'device_type',
    'accuracy_rating', 'clarity_before', 'clarity_after', 'properties'
}
missing = required.difference(events.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

events['occurred_at'] = pd.to_datetime(events['occurred_at'], utc=True, errors='coerce')
events.shape

In [ ]:
def parse_properties(value):
    if isinstance(value, dict):
        return value
    if pd.isna(value) or value == '':
        return {}
    try:
        return json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return {}

props = events['properties'].map(parse_properties)
for key in [
    'client_event_id', 'environment', 'question_id', 'selected_option',
    'response_time_ms', 'changed_answer', 'top_role_id', 'second_role_id',
    'third_role_id', 'match_level', 'preferred_role_was_top_1',
    'preferred_role_was_top_3'
]:
    events[key] = props.map(lambda x: x.get(key))

events['response_time_ms'] = pd.to_numeric(events['response_time_ms'], errors='coerce')
events['is_duplicate'] = events.duplicated('client_event_id', keep='first') & events['client_event_id'].notna()
quality = pd.DataFrame({
    'metric': ['rows', 'sessions', 'missing_session_id', 'missing_environment', 'duplicate_client_event_id'],
    'value': [
        len(events), events['session_id'].nunique(dropna=True),
        events['session_id'].isna().sum(), events['environment'].isna().sum(),
        events['is_duplicate'].sum()
    ]
})
quality

In [ ]:
clean = events.loc[~events['is_duplicate']].copy()
production = clean.loc[clean['environment'].eq('production')].copy()

meaningful_events = {
    'role_opened', 'alternate_role_opened', 'result_alternate_role_opened',
    'role_compare_completed', 'job_opened', 'job_viewed',
    'result_job_card_clicked', 'external_job_clicked', 'result_share_clicked',
    'share_native_completed', 'share_image_downloaded', 'share_link_copied'
}

def has_event(group, names):
    if isinstance(names, str):
        names = {names}
    return group['event_name'].isin(names).any()

rows = []
for session_id, group in production.dropna(subset=['session_id']).groupby('session_id'):
    completed = has_event(group, 'quiz_completed')
    rows.append({
        'session_id': session_id,
        'device_type': group['device_type'].dropna().iloc[0] if group['device_type'].notna().any() else 'unknown',
        'landed': has_event(group, 'landing_viewed'),
        'quiz_started': has_event(group, 'quiz_started'),
        'quiz_completed': completed,
        'result_viewed': has_event(group, 'result_viewed'),
        'meaningful_exploration': completed and has_event(group, meaningful_events),
        'top_role_id': group['top_role_id'].dropna().iloc[-1] if group['top_role_id'].notna().any() else None,
        'accuracy_rating': pd.to_numeric(group['accuracy_rating'], errors='coerce').dropna().iloc[-1]
            if pd.to_numeric(group['accuracy_rating'], errors='coerce').notna().any() else np.nan,
        'clarity_before': pd.to_numeric(group['clarity_before'], errors='coerce').dropna().iloc[-1]
            if pd.to_numeric(group['clarity_before'], errors='coerce').notna().any() else np.nan,
        'clarity_after': pd.to_numeric(group['clarity_after'], errors='coerce').dropna().iloc[-1]
            if pd.to_numeric(group['clarity_after'], errors='coerce').notna().any() else np.nan,
    })

sessions = pd.DataFrame(rows)
sessions['clarity_uplift'] = sessions['clarity_after'] - sessions['clarity_before']
sessions.head()

In [ ]:
funnel = pd.DataFrame({
    'step': ['Landing', 'Quiz started', 'Quiz completed', 'Result viewed', 'Meaningful exploration'],
    'sessions': [
        sessions['landed'].sum(), sessions['quiz_started'].sum(),
        sessions['quiz_completed'].sum(), sessions['result_viewed'].sum(),
        sessions['meaningful_exploration'].sum()
    ]
})
funnel['conversion_from_landing'] = funnel['sessions'] / funnel['sessions'].iloc[0] if len(funnel) and funnel['sessions'].iloc[0] else np.nan
funnel.to_csv(OUTPUT_DIR / 'funnel_overall.csv', index=False)
funnel

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(funnel['step'], funnel['sessions'])
ax.set_title('Data Matters Product Funnel')
ax.set_ylabel('Anonymous sessions')
ax.tick_params(axis='x', rotation=25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'funnel.png', dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
question_events = production.loc[production['event_name'].eq('quiz_question_answered')].copy()
question_events = question_events.loc[question_events['question_id'].notna()]
question_events['valid_response_time'] = question_events['response_time_ms'].between(100, 600000)

question_friction = question_events.groupby('question_id').agg(
    sessions=('session_id', 'nunique'),
    median_response_time_ms=('response_time_ms', 'median'),
    p90_response_time_ms=('response_time_ms', lambda s: s.dropna().quantile(0.90) if s.notna().any() else np.nan),
    answer_change_rate=('changed_answer', lambda s: pd.Series(s).map({True: 1, False: 0, 'true': 1, 'false': 0}).mean())
).reset_index().sort_values('median_response_time_ms', ascending=False)

question_friction.to_csv(OUTPUT_DIR / 'question_friction.csv', index=False)
question_friction

In [ ]:
recommendation_quality = sessions.groupby('top_role_id', dropna=False).agg(
    result_sessions=('session_id', 'nunique'),
    average_accuracy_rating=('accuracy_rating', 'mean'),
    paired_clarity_responses=('clarity_uplift', 'count'),
    average_clarity_uplift=('clarity_uplift', 'mean'),
    meaningful_exploration_rate=('meaningful_exploration', 'mean')
).reset_index().sort_values('result_sessions', ascending=False)

recommendation_quality.to_csv(OUTPUT_DIR / 'recommendation_quality_by_role.csv', index=False)
recommendation_quality

## Interpretation checklist

Before writing a product recommendation:

- confirm the analysis period;
- review duplicate and missing-environment rows;
- report denominators and feedback-response coverage;
- avoid role-level conclusions from tiny samples;
- distinguish descriptive clarity uplift from causal impact;
- connect the finding to a controllable product mechanism.